# DataArts + MRS Spark Big Data ETL Notebook

这个 Notebook 是 VS Code 开发稿，用于演示 Notebook -> Git -> DataArts -> MRS Spark 的开发入口。生产调度脚本已同步到 `spark/jobs/bigdata_lakehouse_etl.py`。

目标链路：读取 OBS raw 层订单/事件数据，执行大数据清洗、去重、脱敏、质量拒绝数据输出，再写入 OBS clean 与 curated 层，并发布 Hive 表和监控事件。

In [ ]:
# Smoke cell: verifies the VS Code Python kernel before connecting to MRS Spark.
import sys
print(sys.version)
print("Kernel ready: Python 3.12 - Huawei DataArts Demo")

## 1. DataArts / MRS runtime parameters

在 DataArts Factory 中，以下参数通过 Spark conf 传入；Notebook 本地开发时可以用这些默认值做代码审查和演示。

In [ ]:
BIGDATA_JOB_CONF = {
    "demo.raw_path": "obs://<bucket>/raw/orders/${biz_date}/",
    "demo.clean_path": "obs://<bucket>/clean/orders/${biz_date}/",
    "demo.reject_path": "obs://<bucket>/clean/rejects/orders/${biz_date}/",
    "demo.curated_path": "obs://<bucket>/curated/daily_channel_metrics/${biz_date}/",
    "demo.event_path": "obs://<bucket>/curated/daily_channel_metrics/${biz_date}/_events",
    "demo.input_format": "csv",
    "demo.biz_date": "2026-06-07",
    "demo.shuffle_partitions": "200",
    "demo.curated_table": "demo_daily_channel_metrics",
}
BIGDATA_JOB_CONF

## 2. PySpark job body

这段脚本适配 MRS Spark：支持 CSV/JSON/Parquet 输入，开启 AQE，按 `biz_date/channel` 分区写出，拒绝数据单独落盘，curated 结果同时写 OBS 和 Hive。

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    count,
    current_timestamp,
    lit,
    regexp_extract,
    round as spark_round,
    sha2,
    sum as spark_sum,
    to_date,
    when,
)


def conf(spark, key, default=None):
    try:
        return spark.conf.get(key)
    except Exception:
        return default


def read_raw(spark, input_format, raw_path):
    fmt = input_format.lower()
    if fmt == "csv":
        return spark.read.option("header", True).option("multiLine", False).csv(raw_path)
    if fmt == "json":
        return spark.read.json(raw_path)
    if fmt == "parquet":
        return spark.read.parquet(raw_path)
    raise ValueError(f"Unsupported input format: {input_format}")


def build_job(spark):
    raw_path = conf(spark, "demo.raw_path")
    clean_path = conf(spark, "demo.clean_path")
    reject_path = conf(spark, "demo.reject_path")
    curated_path = conf(spark, "demo.curated_path")
    event_path = conf(spark, "demo.event_path", f"{curated_path.rstrip('/')}/_events")
    input_format = conf(spark, "demo.input_format", "csv")
    biz_date = conf(spark, "demo.biz_date", "2026-06-07")
    shuffle_partitions = conf(spark, "demo.shuffle_partitions", "200")
    curated_table = conf(spark, "demo.curated_table", "demo_daily_channel_metrics")

    missing = [
        name
        for name, value in {
            "demo.raw_path": raw_path,
            "demo.clean_path": clean_path,
            "demo.reject_path": reject_path,
            "demo.curated_path": curated_path,
        }.items()
        if not value
    ]
    if missing:
        raise ValueError(f"Missing required Spark conf: {', '.join(missing)}")

    spark.conf.set("spark.sql.adaptive.enabled", "true")
    spark.conf.set("spark.sql.shuffle.partitions", shuffle_partitions)

    raw = read_raw(spark, input_format, raw_path).withColumn("biz_date", lit(biz_date))
    parsed = (
        raw.withColumn("amount_number", col("amount").cast("double"))
        .withColumn("event_date", to_date(lit(biz_date)))
        .withColumn("member_hash", sha2(col("member_id").cast("string"), 256))
    )

    valid = (
        parsed.filter(col("order_id").isNotNull())
        .filter(col("amount_number").isNotNull())
        .filter(col("amount_number") > 0)
        .dropDuplicates(["order_id"])
    )

    clean = valid.select(
        "biz_date",
        "event_date",
        "order_id",
        "channel",
        "store_id",
        col("amount_number").alias("amount"),
        "status",
        regexp_extract(col("member_hash"), r"^(.{16})", 1).alias("member_hash_prefix"),
    )

    rejects = (
        parsed.withColumn(
            "reject_reason",
            when(col("order_id").isNull(), lit("missing_order_id"))
            .when(col("amount_number").isNull(), lit("invalid_amount"))
            .when(col("amount_number") <= 0, lit("non_positive_amount"))
            .otherwise(lit(None)),
        )
        .filter(col("reject_reason").isNotNull())
        .select("biz_date", "order_id", "channel", "store_id", "amount", "status", "reject_reason")
    )

    clean.repartition("biz_date", "channel").write.mode("overwrite").partitionBy("biz_date", "channel").parquet(clean_path)
    rejects.repartition("biz_date").write.mode("overwrite").partitionBy("biz_date").parquet(reject_path)

    metrics = (
        clean.groupBy("biz_date", "channel")
        .agg(
            count("*").alias("orders"),
            spark_round(spark_sum(when(col("status") != "refunded", col("amount")).otherwise(0)), 2).alias("revenue"),
            spark_sum(when(col("status") == "refunded", 1).otherwise(0)).alias("refunds"),
        )
        .withColumn("quality_status", when(col("refunds") > 0, lit("review")).otherwise(lit("passed")))
        .withColumn("published_at", current_timestamp())
    )

    metrics.repartition("biz_date").write.mode("overwrite").partitionBy("biz_date").parquet(curated_path)
    spark.sql("CREATE DATABASE IF NOT EXISTS demo_lakehouse")
    metrics.write.mode("overwrite").saveAsTable(f"demo_lakehouse.{curated_table}")

    event = metrics.select(
        "biz_date",
        "channel",
        "orders",
        "revenue",
        "quality_status",
        lit("dataarts-mrs-bigdata-lakehouse-etl").alias("pipeline"),
        current_timestamp().alias("event_time"),
    )
    event.coalesce(1).write.mode("overwrite").json(event_path)


# On MRS, run:
# spark = SparkSession.builder.appName("dataarts-mrs-bigdata-lakehouse-etl").enableHiveSupport().getOrCreate()
# build_job(spark)
# spark.stop()

## 3. DataArts submit template

DataArts 节点提交 MRS Spark 作业时，推荐把 Notebook 中验证过的脚本导出为 `.py`，并使用下面的参数模板。

In [ ]:
spark_submit_template = """
spark-submit \
  --master yarn \
  --deploy-mode cluster \
  --conf demo.raw_path=obs://<bucket>/raw/orders/${biz_date}/ \
  --conf demo.clean_path=obs://<bucket>/clean/orders/${biz_date}/ \
  --conf demo.reject_path=obs://<bucket>/clean/rejects/orders/${biz_date}/ \
  --conf demo.curated_path=obs://<bucket>/curated/daily_channel_metrics/${biz_date}/ \
  --conf demo.event_path=obs://<bucket>/curated/daily_channel_metrics/${biz_date}/_events \
  --conf demo.input_format=csv \
  --conf demo.biz_date=${biz_date} \
  --conf demo.shuffle_partitions=200 \
  spark/jobs/bigdata_lakehouse_etl.py
"""
print(spark_submit_template)